In [3]:
# ============================================================
# TRANSFORM CIRCUITS — LOCAL SILVER LAYER
# ============================================================

import os
from pyspark.sql import functions as F

In [4]:
# -----------------------------------------
# 1. IMPORT ENVIRONMENT CONFIG
# -----------------------------------------
try:
    BRONZE_ROOT
except NameError:
    import nbformat
    from nbconvert import PythonExporter

    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    config_path = os.path.join(project_root, "/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
    with open(config_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/08 23:09:08 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/08 23:09:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 23:09:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/08 23:09:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/08 23:09:09 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/08 23:09:09 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/08 23:09:09 

✔ Spark Session Initialized
✔ Environment Config Loaded
PROJECT_ROOT: /Users/manoharazalki/F1-ANALYTICS


In [5]:
# -----------------------------------------
# 2. Define Bronze + Silver paths
# -----------------------------------------
bronze_file = f"{BRONZE_ROOT}/circuits/circuits.parquet"
silver_folder = f"{SILVER_ROOT}/circuits"
silver_file = f"{silver_folder}/circuits_silver.parquet"

os.makedirs(silver_folder, exist_ok=True)

print("Bronze:", bronze_file)
print("Silver:", silver_file)

Bronze: /Users/manoharazalki/F1-ANALYTICS/data/bronze/circuits/circuits.parquet
Silver: /Users/manoharazalki/F1-ANALYTICS/data/silver/circuits/circuits_silver.parquet


In [6]:
# -----------------------------------------
# 3. Read Bronze Circuits
# -----------------------------------------
circuits_df = spark.read.parquet(bronze_file)
print("Bronze circuits loaded")
circuits_df.show(5)

Bronze circuits loaded
+---------+--------------------+--------------------+--------+--------+----------+---------+--------------------+--------------------+
|circuitId|                 url|         circuitName|     lat|    long|  locality|  country|    ingest_timestamp|         source_file|
+---------+--------------------+--------------------+--------+--------+----------+---------+--------------------+--------------------+
|     NULL|https://en.wikipe...|circuit gilles vi...|    45.5|-73.5228|  montreal|   Canada|2026-06-08 23:07:...|/Users/manoharaza...|
|     NULL|https://en.wikipe...|losail internatio...|   25.49| 51.4542|    lusail|    Qatar|2026-06-08 23:07:...|/Users/manoharaza...|
| adelaide|https://en.wikipe...|adelaide street c...|-34.9272| 138.617|  adelaide|Australia|2026-06-08 23:07:...|/Users/manoharaza...|
| ain-diab|https://en.wikipe...|            ain diab| 33.5786| -7.6875|casablanca|  Morocco|2026-06-08 23:07:...|/Users/manoharaza...|
|  aintree|https://en.wikipe...|

In [7]:
# -----------------------------------------
# 4. Select required columns (drop url)
# -----------------------------------------
circuits_selected_df = circuits_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country").alias("country_name"),
    F.col("ingest_timestamp"),
    F.col("source_file")
)

In [8]:
# -----------------------------------------
# 5. Standardize + rename columns
# -----------------------------------------
circuits_renamed_df = (
    circuits_selected_df
        .withColumnRenamed("circuitId", "circuit_id")
        .withColumnRenamed("circuitName", "circuit_name")
        .withColumnRenamed("lat", "latitude")
        .withColumnRenamed("long", "longitude")
)

In [9]:
# -----------------------------------------
# 6. Business key validation (circuit_id must not be null)
# -----------------------------------------
circuits_valid_df = circuits_renamed_df.filter("circuit_id IS NOT NULL")

In [10]:
# -----------------------------------------
# 7. Remove duplicates
# -----------------------------------------
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])

In [11]:
# -----------------------------------------
# 8. Title Case transformations
# -----------------------------------------
circuits_final_df = (
    circuits_distinct_df
        .withColumn("circuit_name", F.initcap("circuit_name"))
        .withColumn("locality", F.initcap("locality"))
        .withColumn("country_name", F.initcap("country_name"))
)

print("✔ Circuits Silver preview")
circuits_final_df.show(5)

✔ Circuits Silver preview
+-----------+--------------------+--------+---------+----------+------------+--------------------+--------------------+
| circuit_id|        circuit_name|latitude|longitude|  locality|country_name|    ingest_timestamp|         source_file|
+-----------+--------------------+--------+---------+----------+------------+--------------------+--------------------+
|   adelaide|Adelaide Street C...|-34.9272|  138.617|  Adelaide|   Australia|2026-06-08 23:07:...|/Users/manoharaza...|
|   ain-diab|            Ain Diab| 33.5786|  -7.6875|Casablanca|     Morocco|2026-06-08 23:07:...|/Users/manoharaza...|
|    aintree|             Aintree| 53.4769| -2.94056| Liverpool|          Uk|2026-06-08 23:07:...|/Users/manoharaza...|
|albert_park|Albert Park Grand...|-37.8497|  144.968| Melbourne|   Australia|2026-06-08 23:07:...|/Users/manoharaza...|
|   americas|Circuit Of The Am...| 30.1328| -97.6411|    Austin|         Usa|2026-06-08 23:07:...|/Users/manoharaza...|
+-----------+-

In [12]:
# -----------------------------------------
# 9. Write Silver output (Option C)
# -----------------------------------------
(
    circuits_final_df
        .write
        .mode("overwrite")
        .parquet(silver_file)
)

print(f"✔ Circuits Silver written to: {silver_file}")

✔ Circuits Silver written to: /Users/manoharazalki/F1-ANALYTICS/data/silver/circuits/circuits_silver.parquet


In [13]:
# -----------------------------------------
# 10. Read back for validation
# -----------------------------------------
spark.read.parquet(silver_file).show(5)

+-----------+--------------------+--------+---------+----------+------------+--------------------+--------------------+
| circuit_id|        circuit_name|latitude|longitude|  locality|country_name|    ingest_timestamp|         source_file|
+-----------+--------------------+--------+---------+----------+------------+--------------------+--------------------+
|   adelaide|Adelaide Street C...|-34.9272|  138.617|  Adelaide|   Australia|2026-06-08 23:07:...|/Users/manoharaza...|
|   ain-diab|            Ain Diab| 33.5786|  -7.6875|Casablanca|     Morocco|2026-06-08 23:07:...|/Users/manoharaza...|
|    aintree|             Aintree| 53.4769| -2.94056| Liverpool|          Uk|2026-06-08 23:07:...|/Users/manoharaza...|
|albert_park|Albert Park Grand...|-37.8497|  144.968| Melbourne|   Australia|2026-06-08 23:07:...|/Users/manoharaza...|
|   americas|Circuit Of The Am...| 30.1328| -97.6411|    Austin|         Usa|2026-06-08 23:07:...|/Users/manoharaza...|
+-----------+--------------------+------